In [1]:
%pip install -q transformers

In [3]:
%pip install torch

   ---------------------------------------- 0.0/113.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/113.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/113.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/113.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/113.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/113.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/113.7 MB ? eta -:--:--
   ---------------------------------------- 0.8/113.7 MB 884.1 kB/s eta 0:02:08
   ---------------------------------------- 1.0/113.7 MB 986.7 kB/s eta 0:01:55
   ---------------------------------------- 1.3/113.7 MB 1.0 MB/s eta 0:01:48
    --------------------------------------- 1.6/113.7 MB 1.1 MB/s eta 0:01:43
    --------------------------------------- 1.8/113.7 MB 1.2 MB/s eta 0:01:37
    --------------------------------------- 1.8/113.7 MB 1.2 MB/s eta 0:01:37
    ---------------------------

In [14]:
import pandas as pd
import numpy as np
import random
import os
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

from transformers import (
    RobertaTokenizer,
    RobertaModel,
    get_linear_schedule_with_warmup
)

In [5]:
from transformers import RobertaTokenizer, RobertaModel, get_linear_schedule_with_warmup
from torch.optim import AdamW

# transformers >=4.x doesn't export AdamW any more

In [15]:
SEED = 42

def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [16]:
df_train = pd.read_csv("Train.csv")
df_test = pd.read_csv("Test.csv")
submission = pd.read_csv("SampleSubmission.csv")

# Lowercase text
df_train["text"] = df_train["text"].str.lower()
df_test["text"] = df_test["text"].str.lower()

# Remove duplicates
df_train = df_train.drop_duplicates(subset=["text", "label"]).reset_index(drop=True)

Encode Labels

In [17]:
label_map = {
    "Depression": 0,
    "Alcohol": 1,
    "Suicide": 2,
    "Drugs": 3
}

df_train["label"] = df_train["label"].map(label_map)
NUM_CLASSES = 4

In [18]:
class MentalDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=64, is_train=True):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_train = is_train
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        text = self.df.loc[idx, "text"]
        
        encoding = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        
        item = {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten()
        }
        
        if self.is_train:
            item["label"] = torch.tensor(
                self.df.loc[idx, "label"], dtype=torch.long
            )
        
        return item

In [23]:
class MentalModel(nn.Module):
    def __init__(self, num_classes=4):
        super(MentalModel, self).__init__()
        
        self.roberta = RobertaModel.from_pretrained("roberta-base")
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(self.roberta.config.hidden_size, num_classes)
        
    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        pooled = outputs.last_hidden_state[:, 0]
        out = self.dropout(pooled)
        logits = self.fc(out)
        
        return logits

In [24]:
def train_one_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        
        optimizer.zero_grad()
        
        outputs = model(input_ids, attention_mask)
        loss = F.cross_entropy(outputs, labels)
        
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        
    return total_loss / len(loader)

In [25]:
def validate(model, loader):
    model.eval()
    preds = []
    targets = []
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            
            outputs = model(input_ids, attention_mask)
            probs = F.softmax(outputs, dim=1)
            
            preds.append(probs.cpu().numpy())
            targets.append(labels.cpu().numpy())
    
    preds = np.vstack(preds)
    targets = np.concatenate(targets)
    
    return log_loss(targets, preds), preds

Stratified K-Fold Training

In [27]:
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

test_preds = np.zeros((len(df_test), NUM_CLASSES))
val_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(df_train, df_train["label"])):
    
    print(f"\n===== FOLD {fold+1} =====")
    
    train_df = df_train.iloc[train_idx].reset_index(drop=True)
    val_df = df_train.iloc[val_idx].reset_index(drop=True)
    
    train_dataset = MentalDataset(train_df, tokenizer)
    val_dataset = MentalDataset(val_df, tokenizer)
    test_dataset = MentalDataset(df_test, tokenizer, is_train=False)
    
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=16)
    test_loader = DataLoader(test_dataset, batch_size=16)
    
    model = MentalModel(NUM_CLASSES).to(device)
    
    optimizer = AdamW(model.parameters(), lr=2e-5)
    
    total_steps = len(train_loader) * 3
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=0,
        num_training_steps=total_steps
    )
    
    # Train 3 epochs
    for epoch in range(3):
        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler)
        val_loss, _ = validate(model, val_loader)
        print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val LogLoss: {val_loss:.4f}")
    
    val_loss, _ = validate(model, val_loader)
    val_scores.append(val_loss)
    
    # Predict test
    model.eval()
    fold_test_preds = []
    
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            
            outputs = model(input_ids, attention_mask)
            probs = F.softmax(outputs, dim=1)
            fold_test_preds.append(probs.cpu().numpy())
    
    fold_test_preds = np.vstack(fold_test_preds)
    test_preds += fold_test_preds / 5

print("\nCV LogLoss:", np.mean(val_scores))

c:\Users\kawta\anaconda3\envs\tf-env\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kawta\.cache\huggingface\hub\models--roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)



===== FOLD 1 =====


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 203.36it/s, Materializing param=encoder.layer.11.output.dense.weight]              
RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1 | Train Loss: 1.1142 | Val LogLoss: 0.8325
Epoch 2 | Train Loss: 0.7961 | Val LogLoss: 0.6062
Epoch 3 | Train Loss: 0.5753 | Val LogLoss: 0.5199

===== FOLD 2 =====


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 235.68it/s, Materializing param=encoder.layer.11.output.dense.weight]              
RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Epoch 1 | Train Loss: 1.1775 | Val LogLoss: 0.9425
Epoch 2 | Train Loss: 0.8212 | Val LogLoss: 0.6367
Epoch 3 | Train Loss: 0.5689 | Val LogLoss: 0.5616

===== FOLD 4 =====


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 284.51it/s, Materializing param=encoder.layer.11.output.dense.weight]              
RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1 | Train Loss: 1.1148 | Val LogLoss: 0.8820
Epoch 2 | Train Loss: 0.7704 | Val LogLoss: 0.5742
Epoch 3 | Train Loss: 0.4883 | Val LogLoss: 0.4986

===== FOLD 5 =====


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 297.55it/s, Materializing param=encoder.layer.11.output.dense.weight]              
RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1 | Train Loss: 1.1418 | Val LogLoss: 0.8377
Epoch 2 | Train Loss: 0.6473 | Val LogLoss: 0.5282
Epoch 3 | Train Loss: 0.4478 | Val LogLoss: 0.4590

CV LogLoss: 0.4987914932392986


In [28]:
submission["Depression"] = test_preds[:, 0]
submission["Alcohol"] = test_preds[:, 1]
submission["Suicide"] = test_preds[:, 2]
submission["Drugs"] = test_preds[:, 3]

submission.to_csv("submission.csv", index=False)
submission.head()

,ID,Depression,Alcohol,Suicide,Drugs
0,02V56KMO,0.880106,0.023259,0.078441,0.018194
1,03BMGTOK,0.984421,0.002025,0.010372,0.003182
2,03LZVFM6,0.988554,0.001502,0.007299,0.002644
3,0EPULUM5,0.983192,0.001834,0.012020,0.002954
4,0GM4C5GD,0.146268,0.517211,0.091336,0.245185
